Task 3-1

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

# Load and preprocess the Fashion MNIST dataset
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test  = x_test.astype('float32')  / 255.

x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat  = x_test.reshape((x_test.shape[0], -1))

input_dim = 784  # 28 x 28 images
epochs = 20
batch_size = 256

# Function to build autoencoder models with different architectures
def build_autoencoder(hidden_layers, bottleneck_dim):
    input_img = layers.Input(shape=(input_dim,))
    x = input_img

    # Encoder: Build hidden layers leading to the bottleneck
    for units in hidden_layers:
        x = layers.Dense(units, activation='relu')(x)

    # Bottleneck layer
    encoded = layers.Dense(bottleneck_dim, activation='relu')(x)

    # Decoder: Mirror the encoder
    for units in reversed(hidden_layers):
        encoded = layers.Dense(units, activation='relu')(encoded)

    decoded = layers.Dense(input_dim, activation='sigmoid')(encoded)

    autoencoder = models.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

    return autoencoder

# Define different configurations
configurations = [
    {"name": "Vanilla (1 Layer, 64 Neurons)", "hidden_layers": [], "bottleneck": 64},
    {"name": "Vanilla (1 Layer, 256 Neurons)", "hidden_layers": [], "bottleneck": 256},
    {"name": "Deep (2 Layers, 64 Bottleneck)", "hidden_layers": [128], "bottleneck": 64},
    {"name": "Deep (3 Layers, 64 Bottleneck)", "hidden_layers": [256, 128], "bottleneck": 64},
    {"name": "Deep (2 Layers, 256 Bottleneck)", "hidden_layers": [128], "bottleneck": 256},
    {"name": "Custom: Deep (4 Layers, 128 Bottleneck)", "hidden_layers": [512, 256, 128], "bottleneck": 128},  # Custom network
]

histories = {}
models_dict = {}

# Train each model and store the history
for config in configurations:
    print(f"Training {config['name']}...")

    autoencoder = build_autoencoder(config["hidden_layers"], config["bottleneck"])
    history = autoencoder.fit(x_train_flat, x_train_flat,
                              epochs=epochs,
                              batch_size=batch_size,
                              shuffle=True,
                              validation_data=(x_test_flat, x_test_flat),
                              verbose=1)

    histories[config["name"]] = history
    models_dict[config["name"]] = autoencoder

# Plot training and validation loss for comparison
plt.figure(figsize=(10, 6))
for name, history in histories.items():
    plt.plot(history.history['val_loss'], label=name)

plt.title("Validation Loss Across Models")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Compare reconstructions for a few test images
n = 10  # Number of test images to display
test_samples = x_test_flat[:n]

plt.figure(figsize=(20, 12))

# Show original images first row
for j in range(n):
    ax = plt.subplot(len(models_dict) + 1, n, j + 1)
    plt.imshow(x_test_flat[j].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if j == 0:
        ax.set_title("Original", fontsize=10)

# Show reconstructions from each model
for i, (name, model) in enumerate(models_dict.items()):
    reconstructions = model.predict(test_samples)

    for j in range(n):
        ax = plt.subplot(len(models_dict) + 1, n, (i + 1) * n + j + 1)
        plt.imshow(reconstructions[j].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        if j == 0:
            ax.set_title(name, fontsize=10)

plt.suptitle("Comparison of Original vs Reconstructed Images", fontsize=16)
plt.show()
